In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Вказівка параметрів Sampler

<Accordion>
<AccordionItem title="Версії пакетів">

Код на цій сторінці розроблено з такими вимогами.
Рекомендуємо використовувати ці версії або новіші.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Ти можеш використовувати параметри для налаштування примітиву Sampler. Цей розділ зосереджується на тому, як вказувати параметри примітивів Qiskit Runtime. Хоча інтерфейс методу `run()` примітивів є спільним для всіх реалізацій, їхні параметри відрізняються. Ознайомся з відповідними довідниками API для отримання інформації про параметри [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) та [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Налаштування параметрів Sampler
Ти можеш встановлювати параметри під час ініціалізації Sampler, після ініціалізації Sampler або оновлювати їх після ініціалізації. Для інструкцій щодо використання цих технік дивись тему [Вступ до параметрів](/guides/runtime-options-overview#options-precedence).

Крім того, ти можеш встановлювати значення `shots` у методі `run()`, як описано в наступному розділі.
<span id="run-method"></span>
### Метод Run()
Єдині значення, які можна передати до `run()`, це ті, що визначені в інтерфейсі, тобто `shots`. Це перезаписує будь-яке значення, встановлене для `default_shots` для поточного запуску.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Особливі випадки
<span id="shots"></span>
#### Shots

Метод `SamplerV2.run` приймає два аргументи: список PUB-ів, кожен з яких може вказувати специфічне для PUB значення shots, та аргумент-ключове слово shots. Ці значення shots є частиною інтерфейсу виконання Sampler і не залежать від параметрів Sampler Runtime. Вони мають пріоритет над будь-якими значеннями, вказаними як параметри, щоб відповідати абстракції Sampler.

Однак, якщо `shots` не вказано жодним PUB або в аргументі-ключовому слові run (або якщо всі вони мають значення `None`), використовується значення shots з параметрів, зокрема `default_shots`.

Підсумовуючи, ось порядок пріоритету для вказівки shots у Sampler для будь-якого конкретного PUB:

1. Якщо PUB вказує shots, використовуй це значення.
2. Якщо аргумент-ключове слово `shots` вказано у `run`, використовуй це значення.
4. Якщо `twirling` увімкнено (за замовчуванням True), використовується добуток `num_randomizations` та `shots_per_randomization`, вказаний як [параметри `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. Якщо вказано `sampler.options.default_shots`, використовуй це значення.

Таким чином, якщо shots вказані у всіх можливих місцях, використовується те, що має найвищий пріоритет (shots, вказані у PUB).

Приклад: